# Question 1: Research Abstract Clustering

This notebook prepares research abstracts for text analysis, engineers TF-IDF features, and evaluates a K-Means clustering baseline. Run the notebook from top to bottom so that each stage uses the output of the preceding stage.

## 1. Data loading and initial inspection

The dataset is loaded from the project `Dataset` directory. Only the `Abstract` field is used for the text-processing workflow; the preview below confirms that the data has been loaded into a tabular structure.

In [ ]:
from pathlib import Path
import json
import pandas as pd

parent_dir = Path.cwd().parents[1]
file_path = parent_dir / "Dataset" / "research_abstracts.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

data = data["Abstract"]

In [ ]:
df = pd.Series(data).to_frame(name="Abstract")
display(df.head())

## 2. Text preprocessing

The following steps remove duplicate and missing abstracts, standardise the text, lemmatise words, and remove English stop words. The resulting `stop_removed` column is the final text representation used for exploration and modelling.

### 2.1 Data quality checks

Duplicate records and missing abstracts are removed before text transformation. The printed counts should be zero after each operation.

In [ ]:
df = df.drop_duplicates()
print("Duplicate abstracts remaining:", df["Abstract"].duplicated().sum())

In [ ]:
df = df.dropna()
print("Missing abstracts remaining:", df["Abstract"].isna().sum())

### 2.2 Text normalisation

Punctuation and repeated whitespace are removed, then the cleaned text is converted to lowercase. This makes equivalent terms consistent before tokenisation.

In [ ]:
import re

pattern_one = r'[^\w\s]'
pattern_two = r'\s+'

df['clean_Abstract'] = (
    df['Abstract']
    .str.replace(pattern_one, ' ', regex=True)
    .str.replace(pattern_two, ' ', regex=True)
    .str.strip()
)

In [ ]:
df['clean_Abstract'] = df['clean_Abstract'].apply(str.lower)

### 2.3 Lemmatization

Words are reduced to their base forms using part-of-speech-aware WordNet lemmatization. The required NLTK resources are downloaded when this cell is first run.

In [ ]:
from nltk import pos_tag, word_tokenize
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = word_tokenize(text)
    tagged_words = pos_tag(words)

    lemmas = []

    for word, tag in tagged_words:
        if tag.startswith('J'):
            pos = wordnet.ADJ
        elif tag.startswith('V'):
            pos = wordnet.VERB
        elif tag.startswith('N'):
            pos = wordnet.NOUN
        elif tag.startswith('R'):
            pos = wordnet.ADV
        else:
            pos = wordnet.NOUN

        lemmas.append(lemmatizer.lemmatize(word, pos))

    return ' '.join(lemmas)

In [ ]:
df['text_lemmatized'] = df['clean_Abstract'].apply(lemmatize_text)

### 2.4 Stop-word removal

Common English stop words are removed from the lemmatised abstracts, leaving the terms that are most useful for feature engineering and clustering.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

df['stop_removed'] = df['text_lemmatized'].apply(
    lambda x: ' '.join([w for w in x.split() if w.lower() not in ENGLISH_STOP_WORDS])
)

## 3. Exploratory text summary

The word cloud and frequency table provide a quick interpretation of the vocabulary that remains after preprocessing. The final cell compares corpus size before and after stop-word removal.

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud

In [ ]:
text_data = df["stop_removed"]

combined_string = " ".join(text_data)

wordcloud = WordCloud(
    width=800, 
    height=400, 
    background_color="white", 
    colormap="viridis"
).generate(combined_string)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()

In [ ]:
from collections import Counter

word_counts = Counter(
    " ".join(df["stop_removed"]).split()
)

word_freq = pd.DataFrame(
    word_counts.items(),
    columns=["word", "frequency"]
).sort_values(
    "frequency",
    ascending=False
)

display(word_freq.head(20))

In [ ]:
print("Number of abstracts:", len(df))
print("Total words before preprocessing:",
      df["clean_Abstract"].str.split().str.len().sum())
print("Total words after preprocessing:",
      df["stop_removed"].str.split().str.len().sum())

## 4. Feature engineering with TF-IDF

`stop_removed` is used because it is the final preprocessed representation of each abstract. TF-IDF converts the text into numerical features using unigrams and bigrams; the 500-feature cap controls matrix dimensionality while retaining informative terms.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2),
    min_df=2,
)

tfidf_features = tfidf_vectorizer.fit_transform(df["stop_removed"])

tfidf_df = pd.DataFrame(
    tfidf_features.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=df.index,
)

display(tfidf_df.head())
print("Number of abstracts:", tfidf_features.shape[0])
print("Number of engineered features:", tfidf_features.shape[1])

## 5. Baseline clustering model

K-Means is evaluated with TF-IDF features. The data is split into training and test sets so that silhouette scores can be compared across candidate cluster counts and checked for generalisation.

### 5.1 Prepare the train/test split

An 80/20 split is created with a fixed random seed for reproducibility.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_tfidf = tfidf_features

X_train, X_test = train_test_split(
    X_tfidf,
    test_size=0.20,
    random_state=42,
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Number of features:", X_train.shape[1])

### 5.2 Compare candidate cluster counts

Models with two through eight clusters are fitted on the training data. The table reports training and test silhouette scores, their gap, and training inertia for interpretation.

In [ ]:
cluster_results = []

for k in range(2, 9):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10,
    )

    train_labels = kmeans.fit_predict(X_train)
    test_labels = kmeans.predict(X_test)

    train_silhouette = silhouette_score(X_train, train_labels)
    test_silhouette = silhouette_score(X_test, test_labels)

    cluster_results.append({
        "Clusters": k,
        "Train Silhouette": train_silhouette,
        "Test Silhouette": test_silhouette,
        "Generalization Gap": train_silhouette - test_silhouette,
        "Training Inertia": kmeans.inertia_,
    })

results_df = pd.DataFrame(cluster_results)
display(results_df)

### 5.3 Select and evaluate the baseline

The cluster count with the highest test silhouette score is selected. The final baseline is then fitted and its key metrics are reported.

In [ ]:
best_k = results_df.loc[
    results_df["Test Silhouette"].idxmax(),
    "Clusters",
]

print("Selected number of clusters:", best_k)

baseline_model = KMeans(
    n_clusters=int(best_k),
    random_state=42,
    n_init=10,
)

train_clusters = baseline_model.fit_predict(X_train)
test_clusters = baseline_model.predict(X_test)

final_train_silhouette = silhouette_score(X_train, train_clusters)
final_test_silhouette = silhouette_score(X_test, test_clusters)
generalization_gap = final_train_silhouette - final_test_silhouette

print("\n===== BASELINE MODEL PERFORMANCE =====")
print("Number of clusters:", best_k)
print("Training silhouette score:", round(final_train_silhouette, 4))
print("Test silhouette score:", round(final_test_silhouette, 4))
print("Generalization gap:", round(generalization_gap, 4))
print("Training inertia:", round(baseline_model.inertia_, 4))

### 5.4 Visualise model performance

The chart compares training and test silhouette scores for every candidate cluster count. Similar scores indicate more consistent performance between the split datasets.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    results_df["Clusters"],
    results_df["Train Silhouette"],
    marker="o",
    label="Training",
)

plt.plot(
    results_df["Clusters"],
    results_df["Test Silhouette"],
    marker="o",
    label="Testing",
)

plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.title("Baseline K-Means Performance")
plt.legend()
plt.grid(True)
plt.show()